In [ ]:
# [Cell 1/4] Hardware Accelerator Detection (Colab G4 96GB / A100 / L4 / T4 / Local RTX) & Batch Auto-Scaling
import json
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "onnx", "onnxruntime", "openvino"])

NINJANVR_SECURITY_CLASSES = [
    "person", "face", "car", "motorcycle", "bicycle",
    "truck", "bus", "license_plate", "dog", "cat",
    "bird", "package", "waste_bin", "backpack", "umbrella",
]

def detect_gpu_and_batch_size():
    gpu_name = "CPU"
    vram_gb = 0.0
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            gpu_name = props.name
            vram_gb = round(props.total_memory / (1024 ** 3), 1)
    except Exception:
        pass

    # Auto-scale batch size by VRAM: G4 96GB -> 128, A100 40GB -> 64, RTX 4070 Ti Super / L4 16-24GB -> 32, T4 -> 16, CPU -> 8
    if vram_gb >= 80.0 or "G4" in gpu_name or "6000" in gpu_name:
        batch_size = 128
    elif vram_gb >= 40.0 or "A100" in gpu_name:
        batch_size = 64
    elif vram_gb >= 16.0 or "L4" in gpu_name or "4070" in gpu_name:
        batch_size = 32
    elif vram_gb >= 8.0 or "T4" in gpu_name:
        batch_size = 16
    else:
        batch_size = 8
    return gpu_name, vram_gb, batch_size

GPU_NAME, VRAM_GB, BATCH_SIZE = detect_gpu_and_batch_size()
print(json.dumps({"gpu": GPU_NAME, "vram_gb": VRAM_GB, "batch_size": BATCH_SIZE, "nc": len(NINJANVR_SECURITY_CLASSES)}))


In [ ]:
# [Cell 2/4] Dataset Ingestion (ninjanvr-yolo-dataset.zip / ninjanvr-dataset.zip) & Synthetic CCTV Night-IR Augmentation (35% frames)
import math
import random
import zipfile
from pathlib import Path

MODE = os.environ.get("NINJANVR_TRAIN_MODE", "finetune")  # 'finetune' or 'base'
DATASET_ZIP = os.environ.get("NINJANVR_DATASET_ZIP", "ninjanvr-yolo-dataset.zip")
if not Path(DATASET_ZIP).exists() and Path("ninjanvr-dataset.zip").exists():
    DATASET_ZIP = "ninjanvr-dataset.zip"
elif not Path(DATASET_ZIP).exists() and "google.colab" in sys.modules:
    from google.colab import files  # type: ignore
    print("Upload ninjanvr-yolo-dataset.zip exported from NinjaNVR Dataset Studio:")
    uploaded = files.upload()
    if uploaded:
        DATASET_ZIP = next(iter(uploaded.keys()))

DATASET_DIR = Path("./dataset_workspace")

def rgb_to_night_ir_luma(r: int, g: int, b: int, r2_norm: float, vignette_strength: float = 0.35) -> int:
    # ITU-R BT.601 luma Y = 0.299R + 0.587G + 0.114B + radial IR LED vignette falloff
    y = 0.299 * r + 0.587 * g + 0.114 * b
    vignette = max(0.15, min(1.0, 1.0 - vignette_strength * r2_norm))
    return max(0, min(255, int(round(y * vignette))))

if Path(DATASET_ZIP).exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(DATASET_DIR)
    data_yaml_file = DATASET_DIR / "data.yaml"
    if data_yaml_file.exists():
        lines = [
            f"path: {DATASET_DIR.resolve()}" if ln.strip().startswith("path:") else ln
            for ln in data_yaml_file.read_text(encoding="utf-8").splitlines()
        ]
        data_yaml_file.write_text("\n".join(lines) + "\n", encoding="utf-8")
    train_imgs = list((DATASET_DIR / "images" / "train").glob("*.*")) if (DATASET_DIR / "images" / "train").exists() else []
    val_imgs = list((DATASET_DIR / "images" / "val").glob("*.*")) if (DATASET_DIR / "images" / "val").exists() else []
    print(f"Extracted {DATASET_ZIP} into {DATASET_DIR.resolve()} (train={len(train_imgs)}, val={len(val_imgs)})")
else:
    print(f"Ready for dataset archive {DATASET_ZIP} (mode={MODE}, night_ir_prob=0.35)")


In [ ]:
# [Cell 3/4] Dual-Mode YOLO Trainer (base vs finetune with freeze=10 backbone layers)
MODEL_ARCH = os.environ.get("NINJANVR_MODEL_ARCH", "yolo11s.pt")
IMGSZ = int(os.environ.get("NINJANVR_IMGSZ", "640"))
FREEZE_LAYERS = 10 if MODE == "finetune" else 0
EPOCHS = int(os.environ.get("NINJANVR_EPOCHS", "30" if MODE == "finetune" else "100"))
MODEL_NAME = os.environ.get("NINJANVR_MODEL_NAME", "ninjanvr_security_15cl")

TRAIN_CONFIG = {
    "mode": MODE,
    "model": MODEL_ARCH,
    "data": str((DATASET_DIR / "data.yaml").resolve()),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH_SIZE,
    "freeze": FREEZE_LAYERS,
    "perspective": 0.0005,
    "mosaic": 0.5 if MODE == "finetune" else 1.0,
    "project": "./runs/ninjanvr",
    "name": MODEL_NAME,
    "exist_ok": True,
}
print("Training configuration:", json.dumps(TRAIN_CONFIG, indent=2))

TRAINED_MODEL = None
if (DATASET_DIR / "data.yaml").exists():
    try:
        from ultralytics import YOLO  # type: ignore
        TRAINED_MODEL = YOLO(MODEL_ARCH)
        train_kwargs = {k: v for k, v in TRAIN_CONFIG.items() if k not in ("mode", "model")}
        TRAINED_MODEL.train(**train_kwargs)
    except ImportError:
        print("Ultralytics not installed in this environment; skipping live GPU loop.")


In [ ]:
# [Cell 4/4] Export Static ONNX (opset=17) & OpenVINO FP16 IR (.xml/.bin) + Optional Hot-Reload Upload
import base64
import shutil
import urllib.request

NVR_URL = os.environ.get("NINJANVR_URL", "")
NVR_TOKEN = os.environ.get("NINJANVR_TOKEN", "")

EXPORT_SPEC = {
    "onnx": f"{MODEL_NAME}.onnx",
    "openvino_xml": f"{MODEL_NAME}.xml",
    "openvino_bin": f"{MODEL_NAME}.bin",
    "labelmap": "ninjanvr_security_15cl.txt",
    "opset": 17,
    "input_shape": [1, 3, IMGSZ, IMGSZ],
    "output_shape": [1, 19, 8400 if IMGSZ == 640 else 2100],
    "half": True,
}
Path("ninjanvr_security_15cl.txt").write_text("\n".join(NINJANVR_SECURITY_CLASSES) + "\n", encoding="utf-8")

if TRAINED_MODEL is not None:
    onnx_out = TRAINED_MODEL.export(format="onnx", imgsz=IMGSZ, opset=17, dynamic=False, simplify=True)
    if onnx_out and Path(str(onnx_out)).is_file():
        shutil.copy2(str(onnx_out), EXPORT_SPEC["onnx"])
    ov_out = TRAINED_MODEL.export(format="openvino", imgsz=IMGSZ, half=True, dynamic=False)
    if ov_out and Path(str(ov_out)).is_dir():
        for xml_f in Path(str(ov_out)).glob("*.xml"):
            shutil.copy2(xml_f, EXPORT_SPEC["openvino_xml"])
        for bin_f in Path(str(ov_out)).glob("*.bin"):
            shutil.copy2(bin_f, EXPORT_SPEC["openvino_bin"])
    if NVR_URL and Path(EXPORT_SPEC["onnx"]).is_file():
        payload = {
            "filename": EXPORT_SPEC["onnx"],
            "model_base64": base64.b64encode(Path(EXPORT_SPEC["onnx"]).read_bytes()).decode("ascii"),
            "labelmap_filename": EXPORT_SPEC["labelmap"],
            "labelmap": Path(EXPORT_SPEC["labelmap"]).read_text(encoding="utf-8"),
            "model_type": "yolo-generic",
            "width": IMGSZ,
            "height": IMGSZ,
            "input_tensor": "nchw",
            "input_pixel_format": "rgb",
        }
        headers = {"Content-Type": "application/json"}
        if NVR_TOKEN:
            headers["Authorization"] = f"Bearer {NVR_TOKEN}"
        req = urllib.request.Request(
            f"{NVR_URL.rstrip('/')}/api/detectors/models/upload",
            data=json.dumps(payload).encode("utf-8"),
            headers=headers,
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            print("Hot-reloaded on NinjaNVR:", resp.read().decode("utf-8"))
    elif "google.colab" in sys.modules and Path(EXPORT_SPEC["onnx"]).is_file():
        from google.colab import files  # type: ignore
        files.download(EXPORT_SPEC["onnx"])

print("Export specification:", json.dumps(EXPORT_SPEC, indent=2))
